In [ ]:
# ============================================================
# BRANCH B — STRUCTURAL CONVERSION
# D3 — Occupational Employment and Wages — May 2024
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch B: Structural Conversion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================


!pip -q install pymupdf

import json
import hashlib
import re
import fitz

from pathlib import Path
from google.colab import files

DOCUMENT_ID = "D3"
DOCUMENT_NAME = "Occupational Employment and Wages — May 2024"

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

CONVERSION_METHOD = (
    "PyMuPDF text-block conversion to structurally explicit Markdown"
)

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "240f41978ab36001450941f8d74ca7d58b0c865db6ffb6f72d3fcdc2c438a317"
EXPECTED_PAGE_COUNT = 23

SOURCE_PAGE_START = 1
SOURCE_PAGE_END = 5
REFERENCE_SCOPE_PAGES = list(range(SOURCE_PAGE_START, SOURCE_PAGE_END + 1))

EXPECTED_RECORD_COUNT = 70
REFERENCE_PERIOD = "May 2024"

EXPECTED_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Value",
    "Unit",
    "Reference Period"
]

ALLOWED_UNITS = [
    "workers",
    "million workers",
    "percent",
    "USD"
]

EXPECTED_SCOPE_MARKERS = [
    "OCCUPATIONAL EMPLOYMENT AND WAGES",
    "Production occupations",
    "Architecture and engineering occupations",
    "Building and grounds cleaning and maintenance occupations",
    "Largest occupations",
    "Public sector occupations",
    "May 2024"
]

NARRATIVE_SECTION_HEADINGS = [
    "Production occupations",
    "Architecture and engineering occupations",
    "Building and grounds cleaning and maintenance occupations",
    "Largest occupations",
    "Public sector occupations",
    "Changes to the Occupational Employment and Wage Statistics (OEWS) Data",
    "Introduction of New Metropolitan and Nonmetropolitan Area Definitions",
    "Suspension of Publication of Colorado Occupational Employment and Wage Statistics"
]

OUTPUT_DIR = Path("outputs_D3_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configured:", DOCUMENT_ID, BRANCH)
print("Frozen source SHA-256:", EXPECTED_SOURCE_SHA256)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Fixed Stage 1 records:", EXPECTED_RECORD_COUNT)

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload exactly one original D3 PDF.")

SOURCE_FILE = Path(next(iter(uploaded)))

print("Loaded:", SOURCE_FILE)


In [ ]:
# ============================================================
# 2. Verify source format and frozen SHA-256 identity
# ============================================================

if SOURCE_FILE.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FORMAT}; received {SOURCE_FILE.suffix}"
    )

def calculate_sha256(path, chunk_size=8192):
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            sha.update(chunk)
    return sha.hexdigest()

SOURCE_SHA256 = calculate_sha256(SOURCE_FILE)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

print("Observed SHA-256:", SOURCE_SHA256)
print("Matches frozen source:", SOURCE_HASH_MATCH)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "The uploaded PDF does not match the frozen D3 source identity."
    )


In [ ]:
# ============================================================
# 3. Inspect source-document structure and fixed scope
# ============================================================

pdf_document = fitz.open(SOURCE_FILE)
PAGE_COUNT = len(pdf_document)

if PAGE_COUNT != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

page_character_counts = []
for page_number in range(1, PAGE_COUNT + 1):
    text = pdf_document[page_number - 1].get_text("text")
    page_character_counts.append(len(text.strip()))

MACHINE_READABLE = all(count > 0 for count in page_character_counts)

scope_text = "\n".join(
    pdf_document[p - 1].get_text("text")
    for p in REFERENCE_SCOPE_PAGES
)

scope_marker_checks = {
    marker: marker.casefold() in scope_text.casefold()
    for marker in EXPECTED_SCOPE_MARKERS
}

SOURCE_CHECK = {
    "document_id": DOCUMENT_ID,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "expected_page_count": EXPECTED_PAGE_COUNT,
    "observed_page_count": PAGE_COUNT,
    "page_count_matches": PAGE_COUNT == EXPECTED_PAGE_COUNT,
    "machine_readable_text_detected_on_all_pages": MACHINE_READABLE,
    "characters_per_page": page_character_counts,
    "fixed_extraction_scope_pages": [SOURCE_PAGE_START, SOURCE_PAGE_END],
    "scope_marker_checks": scope_marker_checks,
    "all_scope_markers_present": all(scope_marker_checks.values())
}

print(json.dumps(SOURCE_CHECK, indent=2, ensure_ascii=False))

if not MACHINE_READABLE:
    raise ValueError(
        "D3 is expected to contain machine-readable text. "
        "OCR is not introduced in this Branch B conversion."
    )

if not all(scope_marker_checks.values()):
    raise ValueError("Fixed D3 Stage 1 scope markers were not all recovered.")


In [ ]:
# ============================================================
# 4. Extract ordered text blocks from all 23 pages
# ============================================================

page_blocks = {}
block_audit = []

for page_number in range(1, PAGE_COUNT + 1):
    page = pdf_document[page_number - 1]
    blocks = page.get_text("blocks", sort=True)

    cleaned_page_blocks = []

    for block in blocks:
        x0, y0, x1, y1, text, block_number, block_type = block[:7]

        if not text or not text.strip():
            continue

        item = {
            "page": page_number,
            "block_number": int(block_number),
            "block_type": int(block_type),
            "x0": float(x0),
            "y0": float(y0),
            "x1": float(x1),
            "y1": float(y1),
            "text": text
        }

        cleaned_page_blocks.append(item)
        block_audit.append(item)

    page_blocks[page_number] = cleaned_page_blocks

print("Total retained text blocks:", len(block_audit))
print("Blocks on scope pages:",
      sum(len(page_blocks[p]) for p in REFERENCE_SCOPE_PAGES))

In [ ]:
# ============================================================
# 5. Deterministic structural-markup helpers
# ============================================================

def is_isolated_page_marker(line):
    return False

def canonical_heading_if_exact(line):
    candidate = line.strip().rstrip(":")
    for heading in NARRATIVE_SECTION_HEADINGS:
        if candidate.casefold() == heading.casefold():
            return heading
    return None

def is_bullet_line(line):
    stripped = line.lstrip()
    return stripped.startswith("•") or stripped.startswith("\u2022")

def remove_bullet_marker(line):
    stripped = line.lstrip()
    if stripped.startswith("•"):
        return stripped[1:].lstrip()
    return line.strip()


In [ ]:
# ============================================================
# 6. Convert the complete PDF to structurally explicit Markdown
# ============================================================

markdown_lines = [
    "# Occupational Employment and Wages — May 2024",
    "",
    "> Structural conversion of the complete 23-page source PDF.",
    "> The extraction scope remains the fixed Stage 1 headline narrative scope on pages 1–5.",
    ""
]

retained_source_lines = []
excluded_source_lines = []
heading_events = []
bullet_events = []

for page_number in range(1, PAGE_COUNT + 1):
    markdown_lines.append(f"## Source Page {page_number}")
    markdown_lines.append("")

    for block in page_blocks[page_number]:
        for raw_line in block["text"].splitlines():
            line = raw_line.strip()

            if not line:
                continue

            if is_isolated_page_marker(line):
                excluded_source_lines.append({
                    "page": page_number,
                    "text": line,
                    "reason": "isolated_page_marker"
                })
                continue

            heading = canonical_heading_if_exact(line)

            if heading is not None:
                markdown_lines.append(f"### {heading}")
                markdown_lines.append("")
                retained_source_lines.append({
                    "page": page_number,
                    "text": line,
                    "representation": "recognised_heading"
                })
                heading_events.append({
                    "page": page_number,
                    "source_text": line,
                    "heading": heading
                })
                continue

            if is_bullet_line(line):
                bullet_text = remove_bullet_marker(line)
                markdown_lines.append(f"- {bullet_text}")
                retained_source_lines.append({
                    "page": page_number,
                    "text": line,
                    "representation": "bullet"
                })
                bullet_events.append({
                    "page": page_number,
                    "source_text": line
                })
                continue

            markdown_lines.append(line)
            retained_source_lines.append({
                "page": page_number,
                "text": line,
                "representation": "source_text"
            })

        markdown_lines.append("")

STRUCTURAL_MARKDOWN = "\n".join(markdown_lines).rstrip() + "\n"

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D3_branch_B_structural_markdown.md"
)

REPRESENTATION_PATH.write_text(
    STRUCTURAL_MARKDOWN,
    encoding="utf-8"
)

print("Saved:", REPRESENTATION_PATH)
print("Markdown characters:", len(STRUCTURAL_MARKDOWN))
print("Retained source lines:", len(retained_source_lines))
print("Excluded source lines:", len(excluded_source_lines))
print("Recognised headings:", len(heading_events))
print("Bullet lines:", len(bullet_events))


In [ ]:
# ============================================================
# 7. Conversion-integrity verification
# ============================================================

converted_scope_text = "\n".join(
    line
    for line in STRUCTURAL_MARKDOWN.splitlines()
    if line
)

representation_page_checks = {
    p: f"## Source Page {p}" in STRUCTURAL_MARKDOWN
    for p in range(1, PAGE_COUNT + 1)
}

scope_marker_representation_checks = {
    marker: marker.casefold() in STRUCTURAL_MARKDOWN.casefold()
    for marker in EXPECTED_SCOPE_MARKERS
}

missing_retained_lines = []

for item in retained_source_lines:
    source_line = item["text"].strip()

    if item["representation"] == "recognised_heading":
        represented = f"### {canonical_heading_if_exact(source_line)}"
        if represented not in STRUCTURAL_MARKDOWN:
            missing_retained_lines.append(item)

    elif item["representation"] == "bullet":
        represented = f"- {remove_bullet_marker(source_line)}"
        if represented not in STRUCTURAL_MARKDOWN:
            missing_retained_lines.append(item)

    else:
        if source_line not in STRUCTURAL_MARKDOWN:
            missing_retained_lines.append(item)

CONVERSION_INTEGRITY = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_sha256": SOURCE_SHA256,
    "source_hash_matches_frozen_identity": SOURCE_HASH_MATCH,
    "source_page_count": PAGE_COUNT,
    "converted_page_count": PAGE_COUNT,
    "all_source_pages_retained": all(representation_page_checks.values()),
    "representation_page_checks": representation_page_checks,
    "fixed_stage1_scope_pages": [SOURCE_PAGE_START, SOURCE_PAGE_END],
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "conversion_method":
        CONVERSION_METHOD,
    "scope_markers_preserved": scope_marker_representation_checks,
    "all_scope_markers_preserved":
        all(scope_marker_representation_checks.values()),
    "retained_source_line_count": len(retained_source_lines),
    "excluded_source_line_count": len(excluded_source_lines),
    "excluded_source_lines": excluded_source_lines,
    "retained_source_lines_missing_from_representation":
        len(missing_retained_lines),
    "recognised_heading_count": len(heading_events),
    "bullet_event_count": len(bullet_events),
    "ocr_applied": False,
    "page_cropping_applied": False,
    "out_of_scope_pages_removed": False,
    "manual_correction_applied": False,
    "manual_reconstruction_applied": False,
    "semantic_harmonisation_applied": False,
    "value_modification_applied": False,
    "value_rounding_applied": False,
    "derived_calculation_applied": False,
    "conversion_integrity_passed": (
        all(representation_page_checks.values())
        and all(scope_marker_representation_checks.values())
        and len(missing_retained_lines) == 0
        and len(excluded_source_lines) == 0
    ),
    "notes": (
        "The complete 23-page source text is retained. Structural markup "
        "exposes page boundaries, exact recognised section headings and "
        "bullet starts. No standalone numerical source lines are removed."
    )
}

CONVERSION_INTEGRITY_PATH = (
    OUTPUT_DIR / "D3_branch_B_conversion_integrity.json"
)

with open(CONVERSION_INTEGRITY_PATH, "w", encoding="utf-8") as f:
    json.dump(CONVERSION_INTEGRITY, f, indent=2, ensure_ascii=False)

print(json.dumps(CONVERSION_INTEGRITY, indent=2, ensure_ascii=False))

if not CONVERSION_INTEGRITY["conversion_integrity_passed"]:
    raise ValueError("D3 Branch B conversion integrity checks failed.")


In [ ]:
# ============================================================
# 8. Preserve conversion audit and representation metadata
# ============================================================

BLOCK_AUDIT_PATH = OUTPUT_DIR / "D3_branch_B_source_block_audit.json"
LINE_AUDIT_PATH = OUTPUT_DIR / "D3_branch_B_line_conversion_audit.json"

with open(BLOCK_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(block_audit, f, indent=2, ensure_ascii=False)

with open(LINE_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "retained_source_lines": retained_source_lines,
        "excluded_source_lines": excluded_source_lines,
        "heading_events": heading_events,
        "bullet_events": bullet_events
    }, f, indent=2, ensure_ascii=False)

REPRESENTATION_SHA256 = calculate_sha256(REPRESENTATION_PATH)

REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "representation_type":
        "Complete PDF converted to structurally explicit Markdown",
    "source_file": SOURCE_FILE.name,
    "branch_name":
        BRANCH_NAME,

    "conversion_method":
        CONVERSION_METHOD,

    "llm_input_representation":
        "Structural Markdown",
    "source_format": SOURCE_FILE.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "source_pages": PAGE_COUNT,
    "converted_pages": PAGE_COUNT,
    "fixed_extraction_scope_pages": [SOURCE_PAGE_START, SOURCE_PAGE_END],
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "page_extraction_applied": False,
    "out_of_scope_content_retained": True,
    "structural_operations": [
        "Recover machine-readable PDF text blocks in reading order",
        "Expose all source page boundaries",
        "Expose exact recognised narrative section headings",
        "Expose source bullet markers as Markdown bullets"
    ],
    "operations_explicitly_not_applied": [
        "OCR",
        "Physical cropping to pages 1–5",
        "Removal of the Technical Note or Table 1",
        "Semantic label harmonisation",
        "Value calculation or repair",
        "Value precision increase",
        "Unit harmonisation"
    ],
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"]
}

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D3_branch_B_representation.json"
)

with open(REPRESENTATION_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(REPRESENTATION_METADATA, f, indent=2, ensure_ascii=False)

print(json.dumps(REPRESENTATION_METADATA, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 9. Define the fixed extraction schema
# ============================================================

EXPECTED_OUTPUT_STRUCTURE = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "records": [
        {
            "Section": None,
            "Indicator": None,
            "Occupation or Group": None,
            "Value": None,
            "Unit": None,
            "Reference Period": None
        }
    ]
}

print(json.dumps(EXPECTED_OUTPUT_STRUCTURE, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 10. Operationalise the fixed Stage 1 extraction task
# ============================================================

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every explicitly stated occupational statistic from the headline
narrative sections corresponding to source pages 1–5 of the attached
structurally converted Markdown document.

Return one record for every statistic represented within the defined
scope.

For each record, extract:

- Section
- Indicator
- Occupation or Group
- Value
- Unit
- Reference Period

Scope and extraction rules:

- Treat the attached structurally converted Markdown document as the
  only source of information.
- Use only the headline narrative sections corresponding to source
  pages 1–5.
- Extract only information explicitly supported by the document.
- Do not extract the release identifier, release date, contact details,
  website addresses or other publication metadata.
- Do not extract general programme-description counts.
- Do not extract the Technical Note.
- Do not extract the full multi-page Table 1.
- Do not create separate observations from charts when the same values
  are already stated in the narrative.
- Do not include headings without numerical observations as records.
- Do not calculate, infer, reconstruct, aggregate, correct or invent
  any value.
- Do not increase the precision of rounded values.
- Preserve rounded employment values in the reported scale. For example,
  "8.7 million" must be returned as Value 8.7 with Unit
  "million workers".
- Return exact employment counts as numerical values with Unit
  "workers".
- Return employment shares and concentration values as numerical
  values with Unit "percent".
- Return annual mean wages as numerical values with Unit "USD".
- Preserve the occupation, group, industry or location wording used in
  the headline narrative.
- Use "May 2024" as the Reference Period for every record.
- Return Value as a numerical value, not as formatted text.
- Use null only when a requested value is not available.
- Verify that only the defined headline narrative scope has been
  processed.
- Verify that every explicitly stated occupational statistic within
  that scope has been processed.
- Verify that rounded values remain in their reported scale.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
""".strip()

print(EXTRACTION_TASK)


In [ ]:
# ============================================================
# 11. Construct and preserve the Branch B prompt
# ============================================================

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(EXPECTED_OUTPUT_STRUCTURE, indent=2, ensure_ascii=False)}

The complete structurally converted Markdown representation of the
23-page source document is attached.

The extraction scope remains restricted to the headline narrative
sections corresponding to source pages 1–5.

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D3_branch_B_prompt.txt"
PROMPT_PATH.write_text(FULL_PROMPT, encoding="utf-8")

PROMPT_SHA256 = calculate_sha256(PROMPT_PATH)

print(FULL_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# ============================================================
# 12. Create Branch B experiment metadata
# ============================================================

EXPERIMENT_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_FILE.name,
    "source_format": SOURCE_FILE.suffix.lower(),
    "source_sha256": SOURCE_SHA256,
    "source_page_count": PAGE_COUNT,
    "source_machine_readable": MACHINE_READABLE,
    "llm_input_representation":
        "Structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "conversion_method":
        CONVERSION_METHOD,

    "content_validation_performed":
        False,
    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "page_extraction_applied": False,
    "out_of_scope_content_retained": True,
    "normalisation_applied": False,
    "manual_reconstruction_applied": False,
    "manual_correction_applied": False,
    "semantic_harmonisation_applied": False,
    "value_modification_applied": False,
    "value_rounding_applied": False,
    "derived_calculation_applied": False,
    "fixed_extraction_scope_pages": [SOURCE_PAGE_START, SOURCE_PAGE_END],
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "expected_fields": EXPECTED_FIELDS,
    "allowed_units": ALLOWED_UNITS,
    "reference_period": REFERENCE_PERIOD,
    "conversion_integrity_file": CONVERSION_INTEGRITY_PATH.name,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,
    "expected_output_format": "JSON",
    "execution_environment": "Independent ChatGPT conversation",
    "notes": (
        "Only the representation pathway changes relative to Branch A. "
        "The complete 23-page source is retained in structural Markdown, "
        "while the fixed extraction task remains restricted to headline "
        "narrative content on source pages 1–5. No Stage 1 reference "
        "values are supplied to the model. Content-level validation is "
        "performed separately in Validation B — D3."
    )
}

METADATA_PATH = OUTPUT_DIR / "D3_branch_B_experiment_metadata.json"

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_METADATA, f, indent=2, ensure_ascii=False)

print(json.dumps(EXPERIMENT_METADATA, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 13. Download files for independent LLM execution
# ============================================================

for path in [
    REPRESENTATION_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    METADATA_PATH
]:
    files.download(path)

print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D3_branch_B_structural_markdown.md.\n"
    "3. Submit the exact D3_branch_B_prompt.txt content once.\n"
    "4. Do not upload the original PDF or Stage 1 reference values.\n"
    "5. Do not manually repair, correct or regenerate the response.\n"
    "6. Save the complete response exactly as returned."
)


In [ ]:
# ============================================================
# 14. Upload the untouched Branch B model response
# ============================================================

uploaded_output = files.upload()

if len(uploaded_output) != 1:
    raise ValueError(
        "Upload exactly one file containing the complete D3 Branch B response."
    )

UPLOADED_RAW_OUTPUT = Path(next(iter(uploaded_output)))

print("Uploaded:", UPLOADED_RAW_OUTPUT)


In [ ]:
# ============================================================
# 15. Preserve the raw response before parsing
# ============================================================

RAW_RESPONSE_PATH = OUTPUT_DIR / "D3_branch_B_raw_response.txt"

raw_response_text = UPLOADED_RAW_OUTPUT.read_text(encoding="utf-8")
RAW_RESPONSE_PATH.write_text(raw_response_text, encoding="utf-8")

RAW_RESPONSE_SHA256 = calculate_sha256(RAW_RESPONSE_PATH)

print("Raw response preserved.")
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


In [ ]:
# ============================================================
# 16. Parse raw response without modifying it
# ============================================================

valid_json = False
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(raw_response_text)
    valid_json = True
except json.JSONDecodeError as error:
    json_error = str(error)

print("Valid JSON:", valid_json)
if json_error:
    print("JSON parsing error:", json_error)


In [ ]:
# ============================================================
# 17. Validate top-level output structure
# ============================================================

top_level_object_valid = valid_json and isinstance(raw_extraction, dict)

document_id_present = (
    top_level_object_valid and "document_id" in raw_extraction
)
document_id_correct = (
    document_id_present and raw_extraction.get("document_id") == DOCUMENT_ID
)

branch_present = top_level_object_valid and "branch" in raw_extraction
branch_correct = branch_present and raw_extraction.get("branch") == BRANCH

records_present = top_level_object_valid and "records" in raw_extraction
records_is_list = (
    records_present and isinstance(raw_extraction.get("records"), list)
)

records = raw_extraction["records"] if records_is_list else []

TOP_LEVEL_CHECK = {
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_present": bool(document_id_present),
    "document_id_correct": bool(document_id_correct),
    "branch_present": bool(branch_present),
    "branch_correct": bool(branch_correct),
    "records_present": bool(records_present),
    "records_is_list": bool(records_is_list)
}

print(json.dumps(TOP_LEVEL_CHECK, indent=2))


In [ ]:
# ============================================================
# 18. Validate record schemas, field types, units and periods
# ============================================================

record_structure_issues = []
field_type_issues = []
unit_issues = []
reference_period_issues = []

for record_index, record in enumerate(records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(expected_fields - actual_fields)
    additional_fields = sorted(actual_fields - expected_fields)

    if missing_fields or additional_fields:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "additional_fields": additional_fields
        })

    for field in [
        "Section",
        "Indicator",
        "Occupation or Group",
        "Unit",
        "Reference Period"
    ]:
        value = record.get(field)
        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__
            })

    value = record.get("Value")
    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        )
    ):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Value",
            "observed_type": type(value).__name__,
            "observed_value": value
        })

    unit = record.get("Unit")
    if isinstance(unit, str) and unit not in ALLOWED_UNITS:
        unit_issues.append({
            "record_index": record_index,
            "unit": unit
        })

    period = record.get("Reference Period")
    if isinstance(period, str) and period != REFERENCE_PERIOD:
        reference_period_issues.append({
            "record_index": record_index,
            "reference_period": period
        })

print("Record structure issues:", len(record_structure_issues))
print("Field type issues:", len(field_type_issues))
print("Unexpected units:", len(unit_issues))
print("Unexpected reference periods:", len(reference_period_issues))


In [ ]:
# ============================================================
# 19. Scope, duplicate and excluded-content diagnostics
# ============================================================

record_count = len(records)
record_count_valid = record_count == EXPECTED_RECORD_COUNT

def complete_record_key(record):
    if not isinstance(record, dict):
        return None
    return tuple(record.get(field) for field in EXPECTED_FIELDS)

keys = [
    complete_record_key(record)
    for record in records
    if isinstance(record, dict)
]

duplicate_record_key_count = len(keys) - len(set(keys))

excluded_content_markers = [
    "technical note",
    "table 1",
    "media contact",
    "press office",
    "release identifier"
]

excluded_content_issues = []

for record_index, record in enumerate(records):
    if not isinstance(record, dict):
        continue

    descriptive_text = " ".join(
        str(record.get(field, ""))
        for field in ["Section", "Indicator", "Occupation or Group"]
    ).casefold()

    markers = [
        marker for marker in excluded_content_markers
        if marker in descriptive_text
    ]

    if markers:
        excluded_content_issues.append({
            "record_index": record_index,
            "matched_markers": markers
        })

missing_values_by_field = {
    field: sum(
        1
        for record in records
        if not isinstance(record, dict)
        or field not in record
        or record.get(field) is None
    )
    for field in EXPECTED_FIELDS
}

print("Expected records:", EXPECTED_RECORD_COUNT)
print("Observed records:", record_count)
print("Record count valid:", record_count_valid)
print("Duplicate full-record keys:", duplicate_record_key_count)
print("Excluded-content issues:", len(excluded_content_issues))
print("Missing values:", missing_values_by_field)


In [ ]:
# ============================================================
# 20. Save structural/schema diagnostics
# ============================================================

record_schema_valid = (
    len(record_structure_issues) == 0
)

field_types_valid = (
    len(field_type_issues) == 0
)

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid,
    field_types_valid
])

TECHNICAL_DIAGNOSTICS = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "valid_json": bool(valid_json),
    "json_error": json_error,
    **TOP_LEVEL_CHECK,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        int(record_count),
    "record_count_valid":
        bool(record_count_valid),

    "records_with_structure_issues":
        len(record_structure_issues),
    "record_structure_issues":
        record_structure_issues,

    "records_with_type_issues":
        len(field_type_issues),
    "field_type_issues":
        field_type_issues,

    "records_with_unexpected_units":
        len(unit_issues),
    "unit_issues":
        unit_issues,

    "records_with_unexpected_reference_periods":
        len(reference_period_issues),
    "reference_period_issues":
        reference_period_issues,

    "duplicate_record_key_count":
        int(duplicate_record_key_count),

    "excluded_content_issue_count":
        len(excluded_content_issues),
    "excluded_content_issues":
        excluded_content_issues,

    "missing_values_by_field":
        missing_values_by_field,

    "record_schema_valid":
        bool(record_schema_valid),
    "field_types_valid":
        bool(field_types_valid),

    "structurally_evaluable":
        bool(structurally_evaluable)
}

TECHNICAL_DIAGNOSTICS_PATH = OUTPUT_DIR / "D3_branch_B_technical_diagnostics.json"

with open(TECHNICAL_DIAGNOSTICS_PATH, "w", encoding="utf-8") as f:
    json.dump(TECHNICAL_DIAGNOSTICS, f, indent=2, ensure_ascii=False)

print(json.dumps(TECHNICAL_DIAGNOSTICS, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 21. Preserve parsed extraction only when JSON is valid
# ============================================================

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D3_branch_B_parsed_extraction.json"
)

if valid_json:
    with open(PARSED_EXTRACTION_PATH, "w", encoding="utf-8") as f:
        json.dump(raw_extraction, f, indent=2, ensure_ascii=False)

    print("Parsed extraction saved:", PARSED_EXTRACTION_PATH)
else:
    print(
        "No parsed extraction created because the preserved raw response "
        "is invalid JSON."
    )


In [ ]:
# ============================================================
# 22. Create Branch B experiment summary
# ============================================================

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_FILE.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified":
        SOURCE_HASH_MATCH and PAGE_COUNT == EXPECTED_PAGE_COUNT,
    "llm_input_representation":
        "Structural Markdown",
    "representation_file": REPRESENTATION_PATH.name,
    "representation_sha256": REPRESENTATION_SHA256,
    "conversion_method":
        CONVERSION_METHOD,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "content_validation_performed":
        False,
    "conversion_integrity_passed":
        CONVERSION_INTEGRITY["conversion_integrity_passed"],
    "structural_conversion_applied": True,
    "normalisation_applied": False,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "out_of_scope_content_retained": True,
    "fixed_extraction_scope_pages": [SOURCE_PAGE_START, SOURCE_PAGE_END],
    "expected_record_count": EXPECTED_RECORD_COUNT,
    "observed_record_count": int(record_count),
    "valid_json": bool(valid_json),
    "structurally_evaluable": bool(structurally_evaluable),
    "record_count_valid": bool(record_count_valid),
    "records_with_structure_issues": len(record_structure_issues),
    "records_with_type_issues": len(field_type_issues),
    "records_with_unexpected_units": len(unit_issues),
    "records_with_unexpected_reference_periods":
        len(reference_period_issues),
    "duplicate_record_key_count": int(duplicate_record_key_count),
    "excluded_content_issue_count": len(excluded_content_issues),
    "raw_response_preserved": True,
    "raw_response_sha256": RAW_RESPONSE_SHA256,
    "parsed_extraction_created": bool(valid_json),
    "notes":
        "Content-level validation is performed separately in Validation B — D3."
}

SUMMARY_PATH = OUTPUT_DIR / "D3_branch_B_experiment_summary.json"

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(EXPERIMENT_SUMMARY, f, indent=2, ensure_ascii=False)

print(json.dumps(EXPERIMENT_SUMMARY, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# 23. Final artefact inventory and downloads
# ============================================================

generated_outputs = [
    REPRESENTATION_PATH,
    CONVERSION_INTEGRITY_PATH,
    REPRESENTATION_METADATA_PATH,
    BLOCK_AUDIT_PATH,
    LINE_AUDIT_PATH,
    PROMPT_PATH,
    METADATA_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SUMMARY_PATH
]

if valid_json:
    generated_outputs.append(PARSED_EXTRACTION_PATH)

print("Generated D3 Branch B outputs:")
for path in generated_outputs:
    print("-", path.name)

for path in generated_outputs:
    files.download(path)

print(
    "\nFor later Validation B, reuse the frozen D3 Branch A validation "
    "rules unchanged: identity = Section + Indicator + Occupation or Group "
    "+ Reference Period, strict identity first, controlled descriptive "
    "fallback second, Value and Unit excluded from alignment, and the same "
    "numeric/text comparison rules."
)
